# scRNA-seq Gene Expression Analysis

Loads an AnnData `.h5ad` file, explores UMI count distributions, filters cells, and extracts per-cell counts for genes of interest.

**Workflow:**
1. Edit the CONFIG cell below
2. Run cells 1–3 to load data and inspect the UMI distribution
3. Set `MIN_UMI` (and optionally `MAX_UMI`) in the CONFIG cell
4. Run from cell 4 onwards

In [ ]:
# ── CONFIG ── edit these values before running ───────────────────────────────
H5AD_PATH = r"Z:\pipeline\swift-seq\out_dir\workup\results\anndatas\G2swift-1-a-100uMUMI_anndata\starsolo_run_1\GeneFull\adata.h5ad"

GENES_OF_INTEREST = ["Xist", "Spen", "Hdac3"]

MIN_UMI = None   # e.g. 200  — set after inspecting the distribution plot below
MAX_UMI = None   # e.g. 6000 — optional upper cutoff to remove likely doublets

OUTPUT_CSV = "gene_counts_per_cell.csv"
# ─────────────────────────────────────────────────────────────────────────────

In [ ]:
# Cell 1 — Imports
import scanpy as sc
import anndata as ad
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.sparse as sp

sc.settings.verbosity = 1
print(f"scanpy {sc.__version__} | anndata {ad.__version__}")

In [ ]:
# Cell 2 — Load AnnData
adata = sc.read_h5ad(H5AD_PATH)
print(f"Loaded: {adata.n_obs} cells × {adata.n_vars} genes")
print(f"\nadata.obs columns: {list(adata.obs.columns)}")
print(f"adata.var columns: {list(adata.var.columns)}")
print(f"\nFirst 5 gene names (var_names): {list(adata.var_names[:5])}")
print(f"Matrix type: {type(adata.X)}")

In [ ]:
# Cell 3 — UMI distribution (run this before setting MIN_UMI / MAX_UMI)

# Compute total UMI per cell (works for both sparse and dense matrices)
if sp.issparse(adata.X):
    total_counts = np.array(adata.X.sum(axis=1)).flatten()
else:
    total_counts = adata.X.sum(axis=1).flatten()

adata.obs["total_counts"] = total_counts

# Percentile table
percentiles = [1, 5, 10, 25, 50, 75, 90, 95, 99]
pct_values = np.percentile(total_counts, percentiles)
pct_df = pd.DataFrame({"Percentile": percentiles, "UMI count": pct_values.astype(int)})
print(pct_df.to_string(index=False))
print(f"\nMin: {total_counts.min():.0f}  |  Max: {total_counts.max():.0f}  |  Mean: {total_counts.mean():.1f}")

# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(total_counts, bins=100, color="steelblue", edgecolor="none")
axes[0].set_xlabel("Total UMI count per cell")
axes[0].set_ylabel("Number of cells")
axes[0].set_title("UMI count distribution")
if MIN_UMI is not None:
    axes[0].axvline(MIN_UMI, color="red", linestyle="--", label=f"MIN_UMI={MIN_UMI}")
if MAX_UMI is not None:
    axes[0].axvline(MAX_UMI, color="orange", linestyle="--", label=f"MAX_UMI={MAX_UMI}")
if MIN_UMI is not None or MAX_UMI is not None:
    axes[0].legend()

axes[1].hist(np.log10(total_counts + 1), bins=100, color="steelblue", edgecolor="none")
axes[1].set_xlabel("log10(Total UMI count + 1)")
axes[1].set_ylabel("Number of cells")
axes[1].set_title("UMI count distribution (log10 scale)")
if MIN_UMI is not None:
    axes[1].axvline(np.log10(MIN_UMI + 1), color="red", linestyle="--", label=f"MIN_UMI={MIN_UMI}")
if MAX_UMI is not None:
    axes[1].axvline(np.log10(MAX_UMI + 1), color="orange", linestyle="--", label=f"MAX_UMI={MAX_UMI}")
if MIN_UMI is not None or MAX_UMI is not None:
    axes[1].legend()

plt.tight_layout()
plt.show()

if MIN_UMI is None and MAX_UMI is None:
    print("\n>>> Inspect the plots above, then set MIN_UMI (and optionally MAX_UMI) in the CONFIG cell and re-run from Cell 4.")

In [ ]:
# Cell 4 — Filter cells by UMI count
n_before = adata.n_obs

# Capture total counts for ALL cells before the filter overwrites adata
all_total_counts = adata.obs["total_counts"].values.copy()

mask = np.ones(adata.n_obs, dtype=bool)
if MIN_UMI is not None:
    mask &= adata.obs["total_counts"] >= MIN_UMI
if MAX_UMI is not None:
    mask &= adata.obs["total_counts"] <= MAX_UMI

# UMI breakdown (computed before adata is overwritten)
umis_in    = int(all_total_counts[mask].sum())
umis_out   = int(all_total_counts[~mask].sum())
total_umis = umis_in + umis_out

adata  = adata[mask].copy()
n_after = adata.n_obs

print(f"Cells before filtering : {n_before:,}")
print(f"Cells after filtering  : {n_after:,}  ({n_before - n_after:,} removed)")

thresholds = []
if MIN_UMI is not None:
    thresholds.append(f"MIN_UMI={MIN_UMI}")
if MAX_UMI is not None:
    thresholds.append(f"MAX_UMI={MAX_UMI}")
if thresholds:
    print(f"Thresholds applied     : {', '.join(thresholds)}")
else:
    print("No thresholds applied (MIN_UMI and MAX_UMI are both None).")

print()
print(f"UMIs in filtered cells : {umis_in:>12,}  ({100*umis_in/max(total_umis,1):.1f}%)")
print(f"UMIs in excluded cells : {umis_out:>12,}  ({100*umis_out/max(total_umis,1):.1f}%)")
print(f"Total UMIs             : {total_umis:>12,}")

In [ ]:
# Cell 5 — Gene lookup

def find_gene(adata, gene_name):
    """Find gene index by name. Falls back to case-insensitive match."""
    var_names = adata.var_names
    # Exact match
    if gene_name in var_names:
        return gene_name
    # Case-insensitive match
    lower_map = {g.lower(): g for g in var_names}
    if gene_name.lower() in lower_map:
        found = lower_map[gene_name.lower()]
        print(f"  Note: '{gene_name}' not found exactly — using case-insensitive match '{found}'")
        return found
    # Check gene_symbols column if present
    if "gene_symbols" in adata.var.columns:
        sym_map = dict(zip(adata.var["gene_symbols"], var_names))
        if gene_name in sym_map:
            return sym_map[gene_name]
        sym_lower = {k.lower(): v for k, v in sym_map.items()}
        if gene_name.lower() in sym_lower:
            found = sym_lower[gene_name.lower()]
            print(f"  Note: '{gene_name}' matched via gene_symbols column as '{found}'")
            return found
    return None

found_genes = {}
for gene in GENES_OF_INTEREST:
    result = find_gene(adata, gene)
    if result is not None:
        found_genes[gene] = result
        print(f"  [OK]  '{gene}' -> '{result}'")
    else:
        print(f"  [MISSING]  '{gene}' not found in the dataset")

print(f"\nFound {len(found_genes)}/{len(GENES_OF_INTEREST)} genes.")

In [ ]:
# Cell 6 — Extract per-cell counts
counts = {"total_counts": adata.obs["total_counts"].values}

for gene_label, gene_key in found_genes.items():
    gene_data = adata[:, gene_key].X
    if sp.issparse(gene_data):
        gene_data = gene_data.toarray()
    counts[gene_label] = gene_data.flatten()

counts_df = pd.DataFrame(counts, index=adata.obs_names)
counts_df.index.name = "cell_barcode"

print(f"Shape: {counts_df.shape}")
counts_df.head(10)

In [ ]:
# Cell 7 — Summary statistics
rows = []
for gene in found_genes:
    vals = counts_df[gene]
    n_expressing = (vals > 0).sum()
    rows.append({
        "Gene": gene,
        "N cells (total)": len(vals),
        "N cells expressing": n_expressing,
        "% expressing": f"{100 * n_expressing / len(vals):.1f}%",
        "Mean (all cells)": f"{vals.mean():.3f}",
        "Mean (expressing only)": f"{vals[vals > 0].mean():.3f}" if n_expressing > 0 else "N/A",
        "Median (all cells)": f"{vals.median():.1f}",
        "Max": f"{vals.max():.0f}",
    })

summary_df = pd.DataFrame(rows)
print(summary_df.to_string(index=False))

In [ ]:
# Cell 8 — Plots
n_genes = len(found_genes)
if n_genes == 0:
    print("No genes found — skipping plots.")
else:
    fig, axes = plt.subplots(2, n_genes, figsize=(5 * n_genes, 10))
    if n_genes == 1:
        axes = axes.reshape(2, 1)

    colors = ["#e63946", "#457b9d", "#2a9d8f", "#e9c46a"]

    for i, gene in enumerate(found_genes):
        vals = counts_df[gene].values
        color = colors[i % len(colors)]

        # Row 0: violin + strip (all cells)
        ax = axes[0, i]
        parts = ax.violinplot(vals, positions=[0], showmedians=True, showextrema=True)
        for pc in parts["bodies"]:
            pc.set_facecolor(color)
            pc.set_alpha(0.7)
        # Jittered strip
        jitter = np.random.uniform(-0.15, 0.15, size=len(vals))
        ax.scatter(jitter, vals, alpha=0.3, s=4, color=color)
        ax.set_xticks([])
        ax.set_ylabel("UMI count")
        ax.set_title(f"{gene} — all cells (n={len(vals):,})")

        # Row 1: scatter vs total UMI
        ax2 = axes[1, i]
        expressing = vals > 0
        ax2.scatter(
            counts_df["total_counts"][~expressing],
            vals[~expressing],
            alpha=0.3, s=4, color="lightgrey", label="Not expressing"
        )
        ax2.scatter(
            counts_df["total_counts"][expressing],
            vals[expressing],
            alpha=0.6, s=8, color=color, label="Expressing"
        )
        ax2.set_xlabel("Total UMI count")
        ax2.set_ylabel(f"{gene} UMI count")
        ax2.set_title(f"{gene} vs total UMI")
        ax2.legend(fontsize=8, markerscale=2)

    plt.tight_layout()
    plt.show()

In [ ]:
# Cell 9 — Export CSV
import os

out_path = os.path.join(os.getcwd(), OUTPUT_CSV)
counts_df.to_csv(out_path)
print(f"Saved {len(counts_df):,} cells to: {out_path}")